# PubMedQA Retrieval Baselines: BM25 vs Contriever (Expanded Corpus)

**CS505 NLP Final Project**

## 0. Setup

In [ ]:
# Install V100-compatible PyTorch first, then everything else
!pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu118
!pip install datasets transformers faiss-cpu rank_bm25 numpy tqdm psutil

Looking in indexes: https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.1.0 (from versions: 2.2.0+cu118, 2.2.1+cu118, 2.2.2+cu118, 2.3.0+cu118, 2.3.1+cu118, 2.4.0+cu118, 2.4.1+cu118, 2.5.0+cu118, 2.5.1+cu118, 2.6.0+cu118, 2.7.0+cu118, 2.7.1+cu118)
ERROR: No matching distribution found for torch==2.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 85.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
from datasets import load_dataset
from rank_bm25 import BM25Okapi
import torch
import faiss
from tqdm import tqdm
import json
import time
import gc
import psutil

# Verify GPU
print(f"CUDA available: {torch.cuda.is_available()}")

CUDA available: True


## 1. Load PubMedQA + Expanded Corpus

Load the 1,000 labeled queries, then add 10,000 distractor abstracts from the artificial split. This makes retrieval realistically hard — finding 1 passage among 11,000 instead of 1,000.

In [ ]:
# Load labeled split (queries + gold passages)
dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
print(f"Labeled examples: {len(dataset)}")

# Load artificial split for distractor passages
distractors = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")
print(f"Artificial examples available: {len(distractors)}")

Labeled examples: 1000
Artificial examples available: 211269


In [ ]:
# Build queries from labeled split
queries = []
gold_abstracts = []

for example in dataset:
    queries.append(example["question"])
    abstract_sections = example["context"]["contexts"]
    gold_abstracts.append(" ".join(abstract_sections))

print(f"Queries: {len(queries)}")

# Build expanded corpus: gold abstracts + 10K distractors
NUM_DISTRACTORS = 10000

corpus = list(gold_abstracts)  # first 1000 are the gold passages
gold_passage_ids = list(range(len(gold_abstracts)))  # query i -> corpus[i]

distractor_abstracts = []
for i, example in enumerate(distractors):
    if i >= NUM_DISTRACTORS:
        break
    abstract_sections = example["context"]["contexts"]
    distractor_abstracts.append(" ".join(abstract_sections))

corpus.extend(distractor_abstracts)

print(f"Corpus size: {len(corpus)} ({len(gold_abstracts)} gold + {len(distractor_abstracts)} distractors)")
print(f"\nExample query: {queries[0][:120]}...")
print(f"Example passage: {corpus[0][:120]}...")

Queries: 1000
Corpus size: 11000 (1000 gold + 10000 distractors)

Example query: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
Example passage: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascarien...


## 2. Evaluation Metrics

In [ ]:
def evaluate_retrieval(retrieved_ids, gold_ids, k_values=[5, 20]):
    """Compute Recall@k and MRR."""
    recalls = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n = len(gold_ids)

    for i in range(n):
        gold = gold_ids[i]
        retrieved = retrieved_ids[i]

        for rank, doc_id in enumerate(retrieved):
            if doc_id == gold:
                mrr_sum += 1.0 / (rank + 1)
                break

        for k in k_values:
            if gold in retrieved[:k]:
                recalls[k] += 1.0

    results = {}
    for k in k_values:
        results[f"Recall@{k}"] = recalls[k] / n
    results["MRR"] = mrr_sum / n

    return results

## 3. BM25 Baseline

In [ ]:
start_time = time.time()

tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

bm25_retrieved = []
for query in tqdm(queries, desc="BM25 retrieval"):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_ids = np.argsort(scores)[::-1][:20].tolist()
    bm25_retrieved.append(top_ids)

bm25_time = time.time() - start_time
bm25_results = evaluate_retrieval(bm25_retrieved, gold_passage_ids)

print(f"BM25 Results on expanded corpus ({len(corpus)} passages, took {bm25_time:.1f}s):")
for metric, value in bm25_results.items():
    print(f"  {metric}: {value:.4f}")

del bm25, tokenized_corpus
gc.collect()

BM25 retrieval: 100%|██████████| 1000/1000 [00:44<00:00, 22.29it/s]


BM25 Results on expanded corpus (11000 passages, took 45.9s):
  Recall@5: 0.9240
  Recall@20: 0.9530
  MRR: 0.8760
RAM: 24.6% used | Available: 10.3 GB


## 4. Contriever Baseline (GPU)

### Step 1: Load model on GPU

In [ ]:
from transformers import AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def encode_texts(texts, tokenizer, model, batch_size=64, desc="Encoding"):
    """Encode texts using Contriever with mean pooling."""
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        token_embeddings = outputs.last_hidden_state
        attention_mask = inputs["attention_mask"].unsqueeze(-1)
        embeddings = (token_embeddings * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

        del inputs, outputs, token_embeddings, attention_mask, embeddings

    return np.vstack(all_embeddings)

model_name = "facebook/contriever"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.eval()
model.to(device)
print(f"Model loaded on {device}.")

Using device: cuda
Loading facebook/contriever...


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded on cuda.


### Step 2: Encode corpus → save to disk

In [ ]:
contriever_start = time.time()

print(f"Encoding {len(corpus)} corpus passages...")
corpus_embeddings = encode_texts(corpus, tokenizer, model, batch_size=64, desc="Corpus")
np.save("corpus_embeddings_expanded.npy", corpus_embeddings)
print(f"Corpus embeddings shape: {corpus_embeddings.shape} — saved to disk")

del corpus_embeddings
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

Encoding 11000 corpus passages...


Corpus: 100%|██████████| 172/172 [06:08<00:00,  2.14s/it]


Corpus embeddings shape: (11000, 768) — saved to disk


### Step 3: Encode queries → save to disk → free model

In [ ]:
print("Encoding queries...")
query_embeddings = encode_texts(queries, tokenizer, model, batch_size=64, desc="Queries")
np.save("query_embeddings.npy", query_embeddings)
print(f"Query embeddings shape: {query_embeddings.shape} — saved to disk")

del query_embeddings, model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model freed from memory.")

Encoding queries...


Queries: 100%|██████████| 16/16 [00:02<00:00,  6.14it/s]


Query embeddings shape: (1000, 768) — saved to disk
Model freed from memory.


### Step 4: Load embeddings → FAISS search

In [ ]:
corpus_embeddings = np.load("corpus_embeddings_expanded.npy")
query_embeddings = np.load("query_embeddings.npy")
print(f"Loaded corpus: {corpus_embeddings.shape}, queries: {query_embeddings.shape}")

faiss.normalize_L2(corpus_embeddings)
faiss.normalize_L2(query_embeddings)

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

k = 20
scores, retrieved_indices = index.search(query_embeddings, k)

contriever_retrieved = [indices.tolist() for indices in retrieved_indices]
contriever_time = time.time() - contriever_start

contriever_results = evaluate_retrieval(contriever_retrieved, gold_passage_ids)

print(f"\nContriever Results on expanded corpus ({len(corpus)} passages, took {contriever_time:.1f}s):")
for metric, value in contriever_results.items():
    print(f"  {metric}: {value:.4f}")

del corpus_embeddings, query_embeddings, index, scores
gc.collect()

Loaded corpus: (11000, 768), queries: (1000, 768)

Contriever Results on expanded corpus (11000 passages, took 446.4s):
  Recall@5: 0.7930
  Recall@20: 0.9420
  MRR: 0.6970


0

## 5. Summary

In [ ]:
print(f"Corpus size: {len(corpus)} ({len(gold_abstracts)} gold + {NUM_DISTRACTORS} distractors)")
print()
print(f"{'Retriever':<15} {'Recall@5':<12} {'Recall@20':<12} {'MRR':<12}")
print("-" * 51)
print(f"{'BM25':<15} {bm25_results['Recall@5']:<12.3f} {bm25_results['Recall@20']:<12.3f} {bm25_results['MRR']:<12.3f}")
print(f"{'Contriever':<15} {contriever_results['Recall@5']:<12.3f} {contriever_results['Recall@20']:<12.3f} {contriever_results['MRR']:<12.3f}")

print("\n\nLaTeX table rows:")
print(f"BM25 & {bm25_results['Recall@5']:.3f} & {bm25_results['Recall@20']:.3f} & {bm25_results['MRR']:.3f} \\\\")
print(f"Contriever & {contriever_results['Recall@5']:.3f} & {contriever_results['Recall@20']:.3f} & {contriever_results['MRR']:.3f} \\\\")

Corpus size: 11000 (1000 gold + 10000 distractors)

Retriever       Recall@5     Recall@20    MRR         
---------------------------------------------------
BM25            0.924        0.953        0.876       
Contriever      0.793        0.942        0.697       


LaTeX table rows:
BM25 & 0.924 & 0.953 & 0.876 \\
Contriever & 0.793 & 0.942 & 0.697 \\


In [ ]:
results = {
    "bm25": bm25_results,
    "contriever": contriever_results,
    "dataset": "PubMedQA (pqa_labeled)",
    "num_queries": len(queries),
    "num_corpus": len(corpus),
    "num_distractors": NUM_DISTRACTORS,
}
with open("baseline_results_expanded.json", "w") as f:
    json.dump(results, f, indent=2)
print("Results saved to baseline_results_expanded.json")

Results saved to baseline_results_expanded.json


## 6. Error Analysis

In [ ]:
# Contriever failures
failures = []
successes = []
for i in range(len(queries)):
    if gold_passage_ids[i] not in contriever_retrieved[i][:5]:
        failures.append(i)
    else:
        successes.append(i)

print(f"Contriever Recall@5 failures: {len(failures)}/{len(queries)} queries")
print(f"Contriever Recall@5 successes: {len(successes)}/{len(queries)} queries")

print("\n" + "="*60)
print("FAILURE EXAMPLES (gold passage NOT in top 5)")
print("="*60)
for idx in failures[:5]:
    gold_rank = None
    if gold_passage_ids[idx] in contriever_retrieved[idx]:
        gold_rank = contriever_retrieved[idx].index(gold_passage_ids[idx]) + 1
    print(f"\n[Query {idx}] {queries[idx][:120]}...")
    print(f"  Gold passage rank: {gold_rank if gold_rank else 'Not in top 20'}")
    print(f"  Decision: {dataset[idx]['final_decision']}")

print("\n" + "="*60)
print("SUCCESS EXAMPLES (gold passage IN top 5)")
print("="*60)
for idx in successes[:3]:
    gold_rank = contriever_retrieved[idx].index(gold_passage_ids[idx]) + 1
    print(f"\n[Query {idx}] {queries[idx][:120]}...")
    print(f"  Gold passage rank: {gold_rank}")
    print(f"  Decision: {dataset[idx]['final_decision']}")

Contriever Recall@5 failures: 207/1000 queries
Contriever Recall@5 successes: 793/1000 queries

FAILURE EXAMPLES (gold passage NOT in top 5)

[Query 0] Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
  Gold passage rank: 9
  Decision: yes

[Query 4] Can tailored interventions increase mammography use among HMO women?...
  Gold passage rank: 6
  Decision: yes

[Query 5] Double balloon enteroscopy: is it efficacious and safe in a community setting?...
  Gold passage rank: Not in top 20
  Decision: yes

[Query 7] Is adjustment for reporting heterogeneity necessary in sleep disorders?...
  Gold passage rank: 17
  Decision: no

[Query 15] Patient-Controlled Therapy of Breathlessness in Palliative Care: A New Therapeutic Concept for Opioid Administration?...
  Gold passage rank: 7
  Decision: yes

SUCCESS EXAMPLES (gold passage IN top 5)

[Query 1] Landolt C and snellen e acuity: differences in strabismus amblyopia?...
  Gold passage rank: 1
  De

In [ ]:
# BM25 vs Contriever complementary analysis
bm25_wins = []
contriever_wins = []

for i in range(len(queries)):
    bm25_hit = gold_passage_ids[i] in bm25_retrieved[i][:5]
    contriever_hit = gold_passage_ids[i] in contriever_retrieved[i][:5]

    if bm25_hit and not contriever_hit:
        bm25_wins.append(i)
    elif contriever_hit and not bm25_hit:
        contriever_wins.append(i)

print(f"BM25 finds gold in top-5 but Contriever doesn't: {len(bm25_wins)} queries")
print(f"Contriever finds gold in top-5 but BM25 doesn't: {len(contriever_wins)} queries")

if bm25_wins:
    print("\nExamples where BM25 wins:")
    for idx in bm25_wins[:3]:
        print(f"  [{idx}] {queries[idx][:100]}...")

if contriever_wins:
    print("\nExamples where Contriever wins:")
    for idx in contriever_wins[:3]:
        print(f"  [{idx}] {queries[idx][:100]}...")

BM25 finds gold in top-5 but Contriever doesn't: 166 queries
Contriever finds gold in top-5 but BM25 doesn't: 35 queries

Examples where BM25 wins:
  [0] Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
  [4] Can tailored interventions increase mammography use among HMO women?...
  [7] Is adjustment for reporting heterogeneity necessary in sleep disorders?...

Examples where Contriever wins:
  [2] Syncope during bathing in infants, a pediatric form of water-induced urticaria?...
  [25] Amblyopia: is visual loss permanent?...
  [72] Is the combination with 2-methoxyestradiol able to reduce the dosages of chemotherapeutices in the t...
